# 3. Insider Risk, eDiscovery, and Audit

## Insider Risk Management

Not all threats come from outside. **Insider risk management** detects risky behavior by employees, contractors, and partners — both malicious and accidental.

### Risk indicators

| Category | Example activities |
|----------|-------------------|
| **Data theft by departing users** | Mass downloads, USB copies, printing before resignation |
| **Data leaks** | Sharing files with personal email, uploading to non-approved cloud storage |
| **Security policy violations** | Visiting malicious sites, disabling antivirus |
| **Patient data misuse** (healthcare) | Accessing records outside normal duties |
| **Priority user violations** | Unusual activity by executives or people with access to sensitive data |

### How it works

1. **Policies** define which activities to monitor and which users to focus on.
2. **Signals** flow in from Microsoft 365, Defender for Endpoint, HR connectors.
3. **Alerts** are generated when activity patterns match risk indicators.
4. **Cases** are created for investigation (analyst reviews the timeline).
5. **Actions** include escalation, referral to legal, or integration with eDiscovery.

### Exam tip
- Insider risk is about **user behavior**, not external attacks.
- It uses **ML** to correlate signals over time (not just single events).
- Content is **pseudonymized** by default — analysts see "User A", not real names, until they escalate.

In [ ]:
import json
from datetime import datetime, timedelta

# Simulate insider risk detection
USER_ACTIVITIES = [
    # Normal user
    {'user': 'User A', 'day': -5, 'activity': 'Downloaded 3 files from SharePoint', 'risk_score': 5},
    {'user': 'User A', 'day': -4, 'activity': 'Shared document with team channel', 'risk_score': 0},
    {'user': 'User A', 'day': -3, 'activity': 'Edited budget spreadsheet', 'risk_score': 0},
    # Departing user with suspicious activity
    {'user': 'User B', 'day': -7, 'activity': 'Submitted resignation (HR signal)', 'risk_score': 20},
    {'user': 'User B', 'day': -5, 'activity': 'Downloaded 150 files from SharePoint', 'risk_score': 40},
    {'user': 'User B', 'day': -4, 'activity': 'Copied 50 files to USB drive', 'risk_score': 45},
    {'user': 'User B', 'day': -3, 'activity': 'Forwarded 30 emails to personal account', 'risk_score': 50},
    {'user': 'User B', 'day': -2, 'activity': 'Accessed files outside normal work pattern (2 AM)', 'risk_score': 35},
    # Data leak
    {'user': 'User C', 'day': -3, 'activity': 'Uploaded confidential doc to personal Dropbox', 'risk_score': 60},
    {'user': 'User C', 'day': -1, 'activity': 'Shared file with external email (competitor domain)', 'risk_score': 70},
]

# Calculate cumulative risk per user
from collections import defaultdict
user_risk = defaultdict(lambda: {'activities': [], 'total_risk': 0})
for act in USER_ACTIVITIES:
    user_risk[act['user']]['activities'].append(act)
    user_risk[act['user']]['total_risk'] += act['risk_score']

RISK_THRESHOLDS = {'low': 30, 'medium': 80, 'high': 120}

print('=== Insider Risk Management Dashboard ===\n')
for user, data in sorted(user_risk.items(), key=lambda x: -x[1]['total_risk']):
    risk = data['total_risk']
    if risk >= RISK_THRESHOLDS['high']:
        level, icon = 'HIGH', '🔴'
    elif risk >= RISK_THRESHOLDS['medium']:
        level, icon = 'MEDIUM', '🟡'
    elif risk >= RISK_THRESHOLDS['low']:
        level, icon = 'LOW', '🟢'
    else:
        level, icon = 'NONE', '⬜'

    print(f'{icon} {user} — Risk: {level} (score: {risk})')
    for act in data['activities']:
        day_str = f"Day {act['day']:+d}"
        print(f'   {day_str}  {act["activity"]}')
    
    if level in ('HIGH', 'MEDIUM'):
        print(f'   📋 RECOMMENDED ACTION: Create investigation case')
    print()

---
## eDiscovery

**eDiscovery** (electronic discovery) is the process of finding, preserving, and analyzing electronic information for legal cases, investigations, or regulatory requests.

### Three tiers in Microsoft Purview

| Tier | What it includes | License |
|------|-----------------|----------|
| **Content search** | Search across Exchange, SharePoint, OneDrive, Teams | E3 |
| **eDiscovery Standard** | Content search + cases, holds, export | E3 |
| **eDiscovery Premium** | Standard + custodian management, review sets, analytics, predictive coding | E5 |

### The eDiscovery workflow

```
1. Identify → Who are the custodians (people involved)?
2. Preserve → Place legal holds on their mailboxes and sites
3. Collect → Search for relevant content across M365
4. Process → De-duplicate, filter, OCR images
5. Review → Attorneys review documents (Premium: AI-assisted)
6. Export → Package for production to opposing counsel
```

### Legal holds

A legal hold **preserves** all content for a custodian, even if they delete it. The user doesn't know their content is on hold. Content is preserved in a hidden `Recoverable Items` folder.

**Exam tip**: eDiscovery is about **legal and regulatory** investigations. Don't confuse it with insider risk management (behavior monitoring) or DLP (prevention).

---
## Audit

Microsoft Purview Audit records user and admin activities across Microsoft 365.

### Two tiers

| | Audit Standard | Audit Premium |
|-|---------------|---------------|
| **Retention** | 180 days | 1 year (extendable to 10 years) |
| **Events** | Core activities | All Standard + high-value events (mail items accessed, mail items sent) |
| **Access** | Search UI | Search UI + API access |
| **License** | E3 | E5 |

### What gets audited

| Service | Example events |
|---------|----------------|
| Exchange | Email sent, mail read, mailbox login |
| SharePoint | File downloaded, shared, deleted |
| Entra ID | User created, password changed, MFA registered |
| Teams | Meeting joined, message sent, channel created |
| Admin | Policy changed, role assigned, eDiscovery search run |

In [ ]:
# Simulate audit log search
AUDIT_LOGS = [
    {'time': '2026-04-16 09:00', 'user': 'alice@contoso.com', 'activity': 'UserLoggedIn', 'service': 'Entra ID', 'detail': 'MFA completed'},
    {'time': '2026-04-16 09:05', 'user': 'alice@contoso.com', 'activity': 'FileDownloaded', 'service': 'SharePoint', 'detail': 'budget-2026.xlsx'},
    {'time': '2026-04-16 09:10', 'user': 'alice@contoso.com', 'activity': 'FileSyncDownloadedFull', 'service': 'OneDrive', 'detail': '47 files synced'},
    {'time': '2026-04-16 09:15', 'user': 'admin@contoso.com', 'activity': 'Add member to role', 'service': 'Entra ID', 'detail': 'bob → Global Admin'},
    {'time': '2026-04-16 09:20', 'user': 'bob@contoso.com',   'activity': 'Set-Mailbox', 'service': 'Exchange', 'detail': 'Forwarding rule added'},
    {'time': '2026-04-16 09:30', 'user': 'carol@contoso.com', 'activity': 'SearchStarted', 'service': 'eDiscovery', 'detail': 'Search: "project alpha"'},
    {'time': '2026-04-16 10:00', 'user': 'system',            'activity': 'DLPRuleMatch', 'service': 'DLP', 'detail': 'SSN detected in email to external'},
]

print('=== Microsoft Purview Audit Log Search ===\n')
print(f'{"Time":<20} {"User":<25} {"Service":<15} {"Activity":<30} {"Detail"}')
print('─' * 120)
for log in AUDIT_LOGS:
    print(f'{log["time"]:<20} {log["user"]:<25} {log["service"]:<15} {log["activity"]:<30} {log["detail"]}')

print('\n💡 In real Purview, you can filter by date range, user, activity type, and service.')
print('   Premium audit retains logs for up to 10 years and includes mail-read events.')

---
## SC-900 Compliance Domain Cheat Sheet

| Concept | Key fact |
|---------|----------|
| **Service Trust Portal** | Microsoft's audit reports (SOC, ISO, etc.) |
| **Compliance Manager** | Your compliance score + improvement actions |
| **Priva** | Privacy risk management + subject rights requests |
| **Sensitive info types** | Pattern matching (SSN, credit cards) |
| **Trainable classifiers** | ML-based content classification |
| **Sensitivity labels** | Classify + encrypt + watermark documents |
| **DLP** | Block/warn when sharing sensitive data |
| **Retention policies** | Keep/delete by age at location level |
| **Retention labels** | Keep/delete individual items, can be records |
| **Insider risk management** | Detect risky user behavior (data theft, leaks) |
| **eDiscovery** | Find + preserve + review data for legal cases |
| **Legal hold** | Preserve all content even if user deletes it |
| **Audit Standard** | 180-day activity logs |
| **Audit Premium** | 1-10 year logs + high-value events (E5) |

---
## You've completed all SC-900 labs!

### Next steps

1. Take the [SC-900 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/security-compliance-and-identity-fundamentals/practice/assessment?assessment-type=practice&assessmentId=11&practice-assessment-type=certification)
2. Review any weak areas in the Microsoft Learn modules
3. Schedule the exam when you're consistently scoring 80%+ on practice tests

### What's next in this repo

- **AZ-500** (Azure Security Engineer Associate) — hands-on Azure security implementation
- **SC-100** (Cybersecurity Architect Expert) — designing security architectures